# GASP — Taxonomy ML EDA
**Gaia Asteroid Spectral Pipeline v1 — Phase 09**

Exploratory analysis for the taxonomy Random Forest classifier.
- 358 labeled asteroids (Mahlke et al. 2022) of 19,190 total (~1.9%)
- Feature set: 16 Gaia reflectance bands + spectral slopes s1–s4
- Goal: classify all 19,190 with confidence threshold ≥ 0.70

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────────
DATA_FILE   = Path('../data/final/gasp_catalog_v1.parquet')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

BANDS     = [374,418,462,506,550,594,638,682,726,770,814,858,902,946,990,1034]
REFL_COLS = [f'refl_{b}' for b in BANDS]
BAND_UM   = [b/1000 for b in BANDS]

# ── broad-complex mapping (Mahlke 2022 taxonomy) ──────────────────────
COMPLEX_MAP = {
    'S':  'S',   # S-type
    'A':  'S',   # A-type (olivine-rich, S-complex)
    'K':  'S',   # K-type (S-complex)
    'L':  'S',   # L-type (S-complex)
    'Q':  'S',   # Q-type (ordinary chondrite-like)
    'C':  'C',   # C-type
    'B':  'C',   # B-type (C-complex)
    'Ch': 'C',   # Ch-type (hydrated C)
    'X':  'X',   # X-type
    'Xe': 'X',   # Xe-type (E-type subset)
    'Xk': 'X',   # Xk-type
    'M':  'X',   # M-type (metallic, X-complex)
    'Mk': 'X',   # Mk-type
    'P':  'X',   # P-type (low-albedo, X-complex)
    'Pk': 'X',   # Pk-type
    'V':  'V',   # V-type (basaltic, Vesta-family)
    'D':  'D',   # D-type (primitive, red)
    'Z':  'D',   # Z-type (primitive, mapped to D)
}

# ── load data ──────────────────────────────────────────────────────────
df = pd.read_parquet(DATA_FILE)
labeled = df[df['taxonomy'].notna()].copy()
labeled['complex'] = labeled['taxonomy'].map(COMPLEX_MAP)

print(f'Total asteroids : {len(df):,}')
print(f'Labeled         : {len(labeled):,} ({len(labeled)/len(df)*100:.1f}%)')
print(f'Complex coverage: {labeled["complex"].notna().sum()} / {len(labeled)}')
print()
print('Taxonomy value_counts():')
print(labeled['taxonomy'].value_counts().to_string())

## 1. Taxonomy Class Distribution

In [ ]:
vc = labeled['taxonomy'].value_counts()
complex_colors = {
    'S': 'tomato', 'C': 'steelblue', 'X': 'goldenrod',
    'V': 'mediumpurple', 'D': 'saddlebrown',
}
bar_colors = [complex_colors.get(COMPLEX_MAP.get(t, ''), 'gray') for t in vc.index]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(vc.index, vc.values, color=bar_colors, edgecolor='white', linewidth=0.5)

for bar, count in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=9)

# legend for complexes
legend_patches = [mpatches.Patch(color=c, label=f'{k}-complex')
                  for k, c in complex_colors.items()]
ax.legend(handles=legend_patches, fontsize=9, loc='upper right')

ax.set_xlabel('Taxonomy Class (Mahlke et al. 2022)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'Taxonomy class distribution — GASP v1 labeled set (n={len(labeled)})',
             fontsize=13)
plt.tight_layout()

out = FIGURES_DIR / '05_taxonomy_class_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 2. PCA of 16 Gaia Bands (labeled only)

In [ ]:
# use only rows with all 16 bands + known complex
pca_df = labeled[labeled[REFL_COLS].notna().all(axis=1) &
                 labeled['complex'].notna()].copy()

X = pca_df[REFL_COLS].values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=3, random_state=42)
comps = pca.fit_transform(X_scaled)
pca_df['PC1'] = comps[:, 0]
pca_df['PC2'] = comps[:, 1]
pca_df['PC3'] = comps[:, 2]

var_exp = pca.explained_variance_ratio_ * 100
print(f'Variance explained: PC1={var_exp[0]:.1f}%, PC2={var_exp[1]:.1f}%, PC3={var_exp[2]:.1f}%')
print(f'N for PCA: {len(pca_df)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

complex_order = ['S', 'C', 'X', 'V', 'D']
markers = {'S': 'o', 'C': 's', 'X': 'D', 'V': '^', 'D': 'P'}
sizes   = {'S': 40,  'C': 40,  'X': 40,  'V': 55,  'D': 55}

for ax, (xc, yc, xlabel, ylabel) in zip(axes, [
    ('PC1', 'PC2', f'PC1 ({var_exp[0]:.1f}%)', f'PC2 ({var_exp[1]:.1f}%)'),
    ('PC1', 'PC3', f'PC1 ({var_exp[0]:.1f}%)', f'PC3 ({var_exp[2]:.1f}%)'),
]):
    for cx in complex_order:
        sub = pca_df[pca_df['complex'] == cx]
        if len(sub) == 0:
            continue
        ax.scatter(sub[xc], sub[yc],
                   color=complex_colors[cx],
                   marker=markers[cx], s=sizes[cx],
                   alpha=0.8, edgecolors='white', linewidths=0.4,
                   label=f'{cx} (n={len(sub)})')
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.legend(fontsize=9, markerscale=0.9)
    ax.grid(True, alpha=0.2)

plt.suptitle('PCA of 16 Gaia reflectance bands — labeled asteroids only',
             fontsize=13)
plt.tight_layout()

out = FIGURES_DIR / '06_pca_gaia_bands.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 3. Spectral Slopes s1–s4 vs. Broad Complex

In [ ]:
slope_cols = ['s1', 's2', 's3', 's4']
slope_labels = {
    's1': 's1 (374–418 nm)',
    's2': 's2 (418–506 nm)',
    's3': 's3 (506–770 nm)',
    's4': 's4 (770–1034 nm)',
}

slope_df = labeled[labeled['complex'].notna()].copy()

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, sc in zip(axes.flatten(), slope_cols):
    groups = [slope_df[slope_df['complex'] == cx][sc].dropna()
              for cx in complex_order]
    n_nonempty = sum(len(g) > 0 for g in groups)
    labels_used = [f'{cx}\n(n={len(g)})'
                   for cx, g in zip(complex_order, groups) if len(g) > 0]
    data_used   = [g for g in groups if len(g) > 0]
    colors_used = [complex_colors[cx]
                   for cx, g in zip(complex_order, groups) if len(g) > 0]

    bp = ax.boxplot(data_used, patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2),
                    whiskerprops=dict(lw=1.2),
                    flierprops=dict(marker='.', markersize=3, alpha=0.5))
    for patch, color in zip(bp['boxes'], colors_used):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_xticklabels(labels_used, fontsize=10)
    ax.set_ylabel(f'{sc} (µm⁻¹)', fontsize=11)
    ax.set_title(slope_labels[sc], fontsize=11)
    ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
    ax.grid(True, axis='y', alpha=0.25)

plt.suptitle('Spectral slopes s1–s4 by broad taxonomic complex\n(Mahlke et al. 2022 labels)',
             fontsize=13)
plt.tight_layout()

out = FIGURES_DIR / '07_slopes_by_complex.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 4. Correlation Matrix: Slopes vs. Complex (ANOVA-style)

In [ ]:
# Encode complex as numeric for correlation
complex_enc = {'S': 0, 'C': 1, 'X': 2, 'V': 3, 'D': 4}
slope_df['complex_num'] = slope_df['complex'].map(complex_enc)

corr_cols = slope_cols + ['complex_num']
corr_labels = [slope_labels[s] for s in slope_cols] + ['Complex (ord.)']

corr_data = slope_df[corr_cols].dropna()
corr_mat  = corr_data.corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_mat.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Pearson r')

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_labels, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(corr_labels, fontsize=9)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr_mat.values[i,j]:.2f}',
                ha='center', va='center', fontsize=9,
                color='white' if abs(corr_mat.values[i,j]) > 0.5 else 'black')

ax.set_title('Correlation matrix: spectral slopes vs. complex\n(labeled asteroids only)',
             fontsize=12)
plt.tight_layout()

out = FIGURES_DIR / '08_slope_complex_correlation.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')
print()
print('Correlations with complex (ordered S=0, C=1, X=2, V=3, D=4):')
print(corr_mat['complex_num'].drop('complex_num').to_string())

## Summary

| Figure | Path |
|--------|------|
| Class distribution | `figures/05_taxonomy_class_distribution.png` |
| PCA Gaia bands | `figures/06_pca_gaia_bands.png` |
| Slopes by complex | `figures/07_slopes_by_complex.png` |
| Slope correlation | `figures/08_slope_complex_correlation.png` |

**Key findings:**
- S-complex dominates (35%), V-type strongly separated in PCA
- s4 (NIR slope 770–1034 nm) shows strongest complex discrimination
- s2/s3 least discriminating between C and X
- Minority classes (A, Ch, B, K, L, Z, Xk, M, P, Q, Mk, Xe, Pk) will be mapped to broad complexes for classifier